## 5. LDA

### 5.1 LDA Grid search

In [ ]:
tokenized_docs = [doc.split() for doc in docs]  # liste de listes de tokens
dictionary = Dictionary(tokenized_docs)

# Coherence
def compute_coherence(topics):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

def compute_topic_diversity(topics, topk=10):
    all_words = [word for topic in topics for word in topic[:topk]]
    return len(set(all_words)) / len(all_words) if all_words else 0

def get_lda_topics(lda_model, feature_names, topk=10):
    topics = []
    for topic_weights in lda_model.components_:
        idx = topic_weights.argsort()[-topk:][::-1]
        topics.append([feature_names[i] for i in idx])
    return topics

# ==== Paramètres à tester ====
search_params = {
    "n_components": range(8,20),   # nombre de topics
    "max_iter": [100]                   # nombre d'itérations
}

ngram_range_list = [(1,1), (1,2)]

results = []
models = {}

# ==== Grid search ====
for n_topics, ngram_range in product(search_params["n_components"], ngram_range_list):
    
    # CountVectorizer
    vectorizer = CountVectorizer(
        stop_words=STOPWORDS,
        ngram_range=ngram_range,
        min_df=0.005,
        max_df=0.7
    )
    X = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    
    # LDA
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        max_iter=100,
        random_state=42,
        learning_method="online"
    )
    lda.fit(X)
    
    # Topics
    topics = get_lda_topics(lda, feature_names)
    
    # Metrics
    coh = compute_coherence(topics)
    div = compute_topic_diversity(topics)
    score = coh * div
    
    # Stockage
    key = (n_topics, ngram_range)
    models[key] = (lda, vectorizer)
    
    results.append({
        "n_topics": n_topics,
        "ngram_range": ngram_range,
        "coherence": coh,
        "diversity": div,
        "score": score
    })
    
    print(f"n_topics={n_topics}, ngram={ngram_range} | coh={coh:.4f}, div={div:.4f}, score={score:.4f}")

# ==== Résultats ====
df_grid = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)
print("\nTop modèles :")
print(df_grid)

n_topics=8, ngram=(1, 1) | coh=0.4450, div=0.7500, score=0.3337
n_topics=8, ngram=(1, 2) | coh=0.4737, div=0.7000, score=0.3316
n_topics=9, ngram=(1, 1) | coh=0.4871, div=0.7111, score=0.3464
n_topics=9, ngram=(1, 2) | coh=0.4873, div=0.6889, score=0.3357
n_topics=10, ngram=(1, 1) | coh=0.5013, div=0.7200, score=0.3609
n_topics=10, ngram=(1, 2) | coh=0.4989, div=0.6800, score=0.3392
n_topics=11, ngram=(1, 1) | coh=0.4604, div=0.7182, score=0.3306
n_topics=11, ngram=(1, 2) | coh=0.5084, div=0.7364, score=0.3744
n_topics=12, ngram=(1, 1) | coh=0.4500, div=0.6917, score=0.3113
n_topics=12, ngram=(1, 2) | coh=0.5091, div=0.7000, score=0.3564
n_topics=13, ngram=(1, 1) | coh=0.4853, div=0.7077, score=0.3435
n_topics=13, ngram=(1, 2) | coh=0.4824, div=0.7154, score=0.3451
n_topics=14, ngram=(1, 1) | coh=0.4531, div=0.7286, score=0.3301
n_topics=14, ngram=(1, 2) | coh=0.4717, div=0.7000, score=0.3302
n_topics=15, ngram=(1, 1) | coh=0.4509, div=0.7000, score=0.3157
n_topics=15, ngram=(1, 2) | c